## Data Cleaning — Shared Base Dataset

This notebook produces a shared, clean base dataset that all group members should import in their own notebooks.  
It **does not** do any analysis or modelling — only structural cleaning and the addition of universally useful derived variables.

## What this notebook does (step by step)
1. Loads both raw CSV files
2. Gives an initial overview of the data
3. Drops uninformative columns (Ressource, Staff)
4. Converts datetime columns to proper types
5. Recodes acute/night operations based on feedback from Rigshospitalet
6. Adds derived delay and duration variables
7. Flags likely data entry errors (extreme values) without deleting them
8. Matches cancelled patients to completed operations
9. Saves the clean datasets as `.csv` files

## How to use in your own notebook
```python
from Helpers import load_clean_data
df_complete, df_cancelled = load_clean_data()
```
This loads the CSVs and automatically restores all datetime columns — no extra setup needed.

---
**Note from Rigshospitalet (important for interpretation):**
- `Forsinkelse (minutter)`: delay in *start* time. Negative = started earlier than planned.
- `Overskredet (minutter)`: total delay in *end* time vs. planned end. Negative = finished earlier than planned overall.
- `Individuel forsinkelse` (derived): delay that happened *during* the operation itself = `Overskredet - Forsinkelse`.
- Night operations (before 08:00) are **not** planned — treat as acute regardless of the `Akut case` label.
- Extreme values (e.g. ±1000 min) are confirmed data entry errors by Rigshospitalet.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 2. Load Raw Data

In [ ]:
df_complete  = pd.read_csv('../Data and descriptions/Case Rigshospitalet - Completed operations.csv',  sep=';', low_memory=False)
df_cancelled = pd.read_csv('../Data and descriptions/Case Rigshospitalet - Cancelled operations.csv', sep=';', low_memory=False)

print(f"Completed operations : {df_complete.shape[0]:>7,} rows, {df_complete.shape[1]} columns")
print(f"Cancelled operations : {df_cancelled.shape[0]:>7,} rows, {df_cancelled.shape[1]} columns")

## 3. Initial Overview
A quick look at shape, column types, and missing values before we touch anything.

In [ ]:
# --- Completed ---
print("=== COMPLETED: first 5 rows ===")
display(df_complete.head())

print("\n=== COMPLETED: column dtypes ===")
print(df_complete.dtypes.value_counts())

In [ ]:
# Missing value summary — only show columns that actually have missing data
def missing_summary(df, name):
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(1)
    summary = pd.DataFrame({'Missing values': missing, 'Percent (%)': pct})
    summary = summary[summary['Missing values'] > 0].sort_values('Percent (%)', ascending=False)
    print(f"\n=== {name}: {len(summary)} columns with missing data ===")
    print(summary.to_string())
    return summary

missing_complete  = missing_summary(df_complete,  'Completed operations')
missing_cancelled = missing_summary(df_cancelled, 'Cancelled operations')

In [ ]:
# Date ranges — useful sanity check
print("Completed  — Dato range:", df_complete['Dato'].min(), "→", df_complete['Dato'].max())
print("Cancelled  — Dato range:", df_cancelled['Dato og tid'].min(), "→", df_cancelled['Dato og tid'].max())

## 4. Drop Uninformative Columns

The `Ressource.*` and `Staff.*` columns are sparse one-hot encodings (>90% missing) indicating which specific resources or staff types were used. They are not useful for general delay analysis and significantly inflate the column count (247 → ~34 columns).

In [ ]:
cols_before = df_complete.shape[1]

df_complete = df_complete.drop(columns=[c for c in df_complete.columns if c.startswith('Ressource')])
df_complete = df_complete.drop(columns=[c for c in df_complete.columns if c.startswith('Staff')])

print(f"Dropped {cols_before - df_complete.shape[1]} columns (Ressource/Staff).")
print(f"Remaining columns: {df_complete.shape[1]}")
print(list(df_complete.columns))

## 5. Convert Datetime Columns

All timestamp columns are stored as strings in the format `YYYY-MM-DD HH:MM:SS,f`. We convert them to proper `datetime` objects so we can do time arithmetic.

In [ ]:
# All datetime columns in the completed dataset
datetime_cols_complete = [
    'Dato',
    'Pt ankommet til hospitalet',
    'Planlagt stue klargøring start',
    'Stue klargøring start',
    'Stue klargjort',
    'Patient på stuen (Planlagt)',
    'Patient på stuen',
    'Anæstesistart',
    'Anæstesi melder klar',
    'Procedure start',
    'Procedure slut',
    'Patient klar til afgang',
    'Patient forlader stuen (Planlagt)',
    'Patient forlader stuen',
    'Stue rengjort (Planlagt)',
    'Stue rengøring start',
    'Stue rengjort',
    'I opvågning',
    'Anæstesistop',
    'Klar til udskrivelse efter opvågning',
    'Patient forlader afdeling'
]

for col in datetime_cols_complete:
    df_complete[col] = pd.to_datetime(df_complete[col], format='%Y-%m-%d %H:%M:%S,%f', errors='coerce')

# Cancelled dataset only has one datetime column
df_cancelled['Dato og tid'] = pd.to_datetime(df_cancelled['Dato og tid'], errors='coerce')

print("Datetime conversion done.")
print(df_complete[datetime_cols_complete].dtypes)

## 6. Recode Acute Cases Based on Rigshospitalet Feedback

**Rigshospitalet confirmed:** There are no planned operations during the night. Any operation with a procedure start before 08:00 should be treated as acute, regardless of the `Akut case (J/N)` label.

We keep the original column untouched and add a corrected column `Akut_korrigeret`.

In [ ]:
# Map original column to 0/1
df_complete['Akut'] = df_complete['Akut case (J/N)'].map({'Ja': 1, 'Nej': 0})

# Extract hour of procedure start
df_complete['Procedure start time'] = df_complete['Procedure start'].dt.hour

# Corrected acute flag: night operations (before 08:00) are treated as acute
# even if the original label says 'Nej'
df_complete['Akut_korrigeret'] = df_complete['Akut'].copy()
night_mask = df_complete['Procedure start time'] < 8
df_complete.loc[night_mask, 'Akut_korrigeret'] = 1

n_recoded = (night_mask & (df_complete['Akut'] == 0)).sum()
print(f"Operations recoded from planned → acute due to night start time: {n_recoded:,}")
print(f"Total acute (original)   : {df_complete['Akut'].sum():,}")
print(f"Total acute (corrected)  : {df_complete['Akut_korrigeret'].sum():,}")

## 7. Add Derived Delay and Duration Variables

These are variables that multiple group members need, so we compute them once here.

| Variable | Formula | Interpretation |
|---|---|---|
| `Individuel forsinkelse` | `Overskredet - Forsinkelse` | Delay that happened *during* the operation itself (not at start) |
| `Faktisk varighed` | `Procedure slut - Procedure start` (min) | How long the operation actually took |
| `Planlagt varighed` | `Patient forlader stuen (Planlagt) - Patient på stuen (Planlagt)` (min) | How long the full slot was planned to take |
| `Varighed afvigelse` | `Faktisk varighed - Planlagt varighed` | Positive = ran over, negative = finished early |
| `Ugedag` | from `Dato` | Day of week |
| `Måned` | from `Dato` | Month number |

In [ ]:
# Delay during the operation itself
# Confirmed by Rigshospitalet:
#   Forsinkelse = delay in START time
#   Overskredet = total delay in END time (includes both start delay and any extra time during the op)
#   Individuel forsinkelse = how much extra time the operation ITSELF took, independent of start delay
df_complete['Individuel forsinkelse'] = df_complete['Overskredet (minutter)'] - df_complete['Forsinkelse (minutter)']

# Actual duration of the procedure in minutes
df_complete['Faktisk varighed'] = (
    df_complete['Procedure slut'] - df_complete['Procedure start']
).dt.total_seconds() / 60

# Planned duration of the full slot (patient enters room → patient leaves room, planned)
df_complete['Planlagt varighed'] = (
    df_complete['Patient forlader stuen (Planlagt)'] - df_complete['Patient på stuen (Planlagt)']
).dt.total_seconds() / 60

# How much the operation deviated from its planned duration
# Positive = ran over planned time, Negative = finished ahead of planned time
df_complete['Varighed afvigelse'] = df_complete['Faktisk varighed'] - df_complete['Planlagt varighed']

# Time features
df_complete['Ugedag'] = df_complete['Dato'].dt.day_name()
df_complete['Måned']  = df_complete['Dato'].dt.month

print("Derived variables added:")
derived_cols = ['Individuel forsinkelse', 'Faktisk varighed', 'Planlagt varighed', 'Varighed afvigelse', 'Ugedag', 'Måned']
display(df_complete[derived_cols].describe())

## 8. Flag Data Entry Errors (Extreme Values)

**Rigshospitalet confirmed** that very large absolute values (e.g. ±1000 min) are data entry errors — not real delays. They suggested a threshold of "a couple of hours" and stated that values like -1000 are never real.

We use ±240 minutes (4 hours) as a conservative threshold for flagging. We **do not delete** these rows — they remain in the dataset. Each person can choose to filter them out in their own notebook using:
```python
df = df_complete[~df_complete['Forsinkelse_fejl_flag']]
```

In [ ]:
OUTLIER_THRESHOLD = 240  # minutes — justified by Rigshospitalet feedback

# Flag rows where Forsinkelse or Overskredet seem like data errors
df_complete['Forsinkelse_fejl_flag'] = (
    (df_complete['Forsinkelse (minutter)'].abs() > OUTLIER_THRESHOLD) |
    (df_complete['Overskredet (minutter)'].abs() > OUTLIER_THRESHOLD)
)

n_flagged = df_complete['Forsinkelse_fejl_flag'].sum()
pct_flagged = n_flagged / len(df_complete) * 100

print(f"Rows flagged as likely data entry errors (|Forsinkelse| or |Overskredet| > {OUTLIER_THRESHOLD} min):")
print(f"  {n_flagged:,} rows ({pct_flagged:.2f}% of total)")
print(f"  {len(df_complete) - n_flagged:,} rows remain after filtering these out")

In [ ]:
# Quick distribution check of the delay columns to validate the flag threshold
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

clean = df_complete[~df_complete['Forsinkelse_fejl_flag']]

axes[0].hist(clean['Forsinkelse (minutter)'].dropna(), bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Forsinkelse (minutter) — after flag filter')
axes[0].set_xlabel('Minutter')
axes[0].set_ylabel('Antal operationer')

axes[1].hist(clean['Overskredet (minutter)'].dropna(), bins=80, color='coral', edgecolor='white')
axes[1].set_title('Overskredet (minutter) — after flag filter')
axes[1].set_xlabel('Minutter')
axes[1].set_ylabel('Antal operationer')

plt.tight_layout()
plt.show()

## 9. Match Cancelled Patients to Completed Operations

**Rigshospitalet confirmed** that IDs can be compared across the two datasets — the same ID = the same patient.

We add a flag `Tidligere aflyst` to `df_complete` indicating whether the patient also appears in the cancelled dataset (i.e. they were cancelled at least once before completing their operation).

In [ ]:
ids_cancelled = set(df_cancelled['ID'])

df_complete['Tidligere aflyst'] = df_complete['Case-ID Anonymous'].isin(ids_cancelled).astype(int)

n_previously_cancelled = df_complete['Tidligere aflyst'].sum()
print(f"Completed operations where patient also appears in cancelled dataset: {n_previously_cancelled:,}")
print(f"({n_previously_cancelled / len(df_complete) * 100:.1f}% of all completed operations)")

## 10. Final Overview of Clean Dataset

In [ ]:
print("=== CLEAN COMPLETED DATASET ===")
print(f"Shape: {df_complete.shape[0]:,} rows × {df_complete.shape[1]} columns")
print(f"\nDate range: {df_complete['Dato'].min().date()} → {df_complete['Dato'].max().date()}")
print(f"Specialties: {df_complete['Speciale'].nunique()} unique")
print(f"Operating rooms (Stue): {df_complete['Stue'].nunique()} unique")
print(f"Acute operations (corrected): {df_complete['Akut_korrigeret'].sum():,} ({df_complete['Akut_korrigeret'].mean()*100:.1f}%)")
print(f"Flagged as data errors: {df_complete['Forsinkelse_fejl_flag'].sum():,} ({df_complete['Forsinkelse_fejl_flag'].mean()*100:.2f}%)")
print(f"Previously cancelled patients: {df_complete['Tidligere aflyst'].sum():,}")
print()
print("=== CLEAN CANCELLED DATASET ===")
print(f"Shape: {df_cancelled.shape[0]:,} rows × {df_cancelled.shape[1]} columns")

In [ ]:
# Summary stats for the key delay variables
delay_cols = ['Forsinkelse (minutter)', 'Overskredet (minutter)', 'Individuel forsinkelse',
              'Faktisk varighed', 'Planlagt varighed', 'Varighed afvigelse']

display(df_complete[delay_cols].describe().round(1))

In [ ]:
# Show all final column names for reference
print("Final columns in df_complete:")
for i, col in enumerate(df_complete.columns):
    print(f"  {i+1:>2}. {col}")

## 11. Save Clean Datasets

Saved as `.csv` files — easy to open in Excel, share, and load in any tool.

**To load in your own notebook:**
```python
import pandas as pd
df_complete  = pd.read_csv('../Data and descriptions/clean_complete.csv',  sep=';')
df_cancelled = pd.read_csv('../Data and descriptions/clean_cancelled.csv', sep=';')
```

In [ ]:
df_complete.to_csv('../Data and descriptions/clean_complete.csv',  sep=';', index=False)
df_cancelled.to_csv('../Data and descriptions/clean_cancelled.csv', sep=';', index=False)

print("Saved:")
print("  ../Data and descriptions/clean_complete.csv")
print("  ../Data and descriptions/clean_cancelled.csv")